In [1]:
!pip install -q -U diffusers accelerate transformers safetensors invisible_watermark

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 40.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 81.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 49.9 MB/s eta 0:00:00


In [2]:
import torch, os, json
from diffusers import StableDiffusionXLPipeline

os.makedirs("/content/output", exist_ok=True)

model_name = "stabilityai/stable-diffusion-xl-base-1.0"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32
device = "cuda" if torch.cuda.is_available() else "cpu"

pipe = StableDiffusionXLPipeline.from_pretrained(
    model_name,
    torch_dtype=torch_dtype,
    use_safetensors=True,
    variant="fp16" if device == "cuda" else None,
)
pipe = pipe.to(device)
pipe.enable_attention_slicing()

positive_magic = ", ultra HD, professional e-commerce product photography, sharp focus, studio lighting"
negative_prompt = "blurry, low quality, cropped, watermark, text, extra limbs, deformed"
print("Model loaded on", device)

Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v0.40.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


model_index.json:   0%|          | 0.00/609 [00:00<?, ?B/s]

Fetching 19 files:   0%|          | 0/19 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Model loaded on cuda


In [3]:
PROMPTS = [
  (1, "cattleya", 1, "Product photo of a Cattleya orchid, classic lavender-purple ruffled petals with a deep magenta lip, single bloom in sharp focus, soft studio lighting, clean white background, e-commerce style"),
  (1, "cattleya", 2, "Cattleya orchid in white with a bright yellow-orange throat, 3-4 blooms on one stem, soft green foliage visible, neutral light-grey backdrop, professional product photography"),
  (1, "cattleya", 3, "Deep red-maroon Cattleya hybrid, velvety petal texture, potted in a 15cm terracotta pot, natural window light, minimalist white studio backdrop"),
  (1, "cattleya", 4, "Pastel pink Cattleya with white edges, delicate ruffled lip, close-up macro shot showing petal texture, softbox lighting, pure white background"),
  (1, "cattleya", 5, "Bicolor Cattleya - orange petals with a fuchsia lip, full plant with 2 stems in bloom, in a black nursery pot, bright even studio lighting, clean backdrop"),
  (2, "cymbidium", 1, "Cymbidium orchid, long arching spray of pale green blooms with a red-spotted lip, in a 20cm pot, elegant vertical composition, soft neutral background"),
  (2, "cymbidium", 2, "Cymbidium hybrid in rich burgundy-red, dense flower spike, glossy strap-like leaves, studio product shot, white background, even lighting"),
  (2, "cymbidium", 3, "Mini Cymbidium in creamy yellow with a maroon lip, compact potted plant in an 8cm pot, top-down and front angle available, clean white backdrop"),
  (2, "cymbidium", 4, "White Cymbidium with a soft pink blush, tall flower spike arching gracefully, potted in a woven basket, bright airy studio lighting"),
  (2, "cymbidium", 5, "Deep bronze/olive-toned Cymbidium, exotic mottled petals, close-up of a single bloom cluster, dark neutral backdrop for contrast, professional lighting"),
  (3, "dendrobium", 1, "Dendrobium orchid spray, classic white with a purple-pink lip, multiple blooms along a slender upright stem, 15cm pot, clean white studio background"),
  (3, "dendrobium", 2, "Dendrobium 'Sonia' hybrid in vivid magenta-pink, dense flower spray, glossy cane visible, bright even lighting, e-commerce product shot"),
  (3, "dendrobium", 3, "Antelope-type Dendrobium in yellow-green with twisted petals, unusual exotic form, macro detail shot, soft neutral background"),
  (3, "dendrobium", 4, "White Dendrobium with a golden-yellow throat, elegant arching spray, potted in a tall narrow pot, soft diffused studio lighting"),
  (3, "dendrobium", 5, "Orange-red Dendrobium hybrid, compact potted plant, full-plant product shot showing foliage and multiple stems, white backdrop"),
  (4, "oncidium", 1, "Oncidium 'Dancing Lady' orchid, cascading spray of small golden-yellow blooms with brown markings, dramatic arching stem, white studio background"),
  (4, "oncidium", 2, "Oncidium hybrid in deep chocolate-brown and yellow, dense branching flower spray, potted plant, warm studio lighting, clean backdrop"),
  (4, "oncidium", 3, "Miniature Oncidium in bright red-orange, compact spray, close-up macro of the ruffled lip detail, soft white background"),
  (4, "oncidium", 4, "Pastel pink and white Oncidium spray, delicate small blooms, full potted plant on a plain surface, bright natural-style lighting"),
  (4, "oncidium", 5, "Fragrant Oncidium 'Sweet Sugar' type, buttery yellow with cinnamon-brown speckles, multiple stems in bloom, professional product photography"),
  (5, "paphiopedilum", 1, "Paphiopedilum orchid, classic green and maroon pouch-shaped bloom, single flower on a short stem, mottled foliage, white studio background"),
  (5, "paphiopedilum", 2, "Paphiopedilum hybrid with a striped white-and-burgundy dorsal sepal, dramatic slipper-shaped lip, close-up product shot, soft lighting"),
  (5, "paphiopedilum", 3, "Multi-flowered Paphiopedilum in deep purple-black tones, exotic dark coloring, potted in a small ceramic pot, neutral grey backdrop"),
  (5, "paphiopedilum", 4, "Yellow-green Paphiopedilum with spotted petals, plain mottled leaves, top-down and side angle composite, clean white background"),
  (5, "paphiopedilum", 5, "Pink-and-white Paphiopedilum, rounded slipper pouch, full plant shot in a 12cm pot, bright even studio lighting"),
  (6, "phalaenopsis", 1, "Classic white Phalaenopsis orchid, 5-6 flat round blooms on an arching spike, glossy dark green leaves, potted, pure white studio background"),
  (6, "phalaenopsis", 2, "Phalaenopsis in deep pink-magenta, full bloom spray, elegant potted plant, soft box lighting, clean e-commerce style backdrop"),
  (6, "phalaenopsis", 3, "Yellow Phalaenopsis with red barring/spots, exotic patterned petals, close-up bloom detail, neutral white background"),
  (6, "phalaenopsis", 4, "Bicolor Phalaenopsis - white petals with a purple lip and throat, double spike in bloom, bright studio lighting"),
  (6, "phalaenopsis", 5, "Mini Phalaenopsis in soft lavender, compact potted plant, full-plant angle showing leaves and roots, clean white backdrop"),
  (7, "vanda", 1, "Vanda orchid, vivid blue-purple checkered/mottled petals, hanging in a slatted wooden basket, roots visible, bright natural studio lighting"),
  (7, "vanda", 2, "Vanda hybrid in hot pink with darker veining, multiple blooms on a tall spike, potted in a hanging basket, clean neutral background"),
  (7, "vanda", 3, "Orange-red Vanda, dense cluster of flat blooms, close-up macro shot of petal pattern, white studio backdrop"),
  (7, "vanda", 4, "White-and-purple Vanda, elegant spray form, full plant with visible aerial roots, soft even studio lighting"),
  (7, "vanda", 5, "Deep burgundy Vanda hybrid, rich velvety texture, potted plant product shot, bright clean white background"),
  (8, "other-orchids", 1, "Brassia 'Spider Orchid', elongated spidery yellow-green petals with brown spots, dramatic form, white studio background"),
  (8, "other-orchids", 2, "Epidendrum 'Star Orchid' in bright orange-red, clustered small star-shaped blooms atop a tall stem, clean neutral backdrop"),
  (8, "other-orchids", 3, "Miltonia 'Pansy Orchid' in pink and white with a face-like pattern, single bloom close-up, soft studio lighting"),
  (8, "other-orchids", 4, "Zygopetalum hybrid, green petals with purple-striped lip, fragrant bloom, potted plant, white background"),
  (8, "other-orchids", 5, "Mixed exotic hybrid orchid spray, unusual multi-tone coloring, full potted plant shot, bright even studio lighting"),
  (9, "granular-fertilizer", 1, "White plastic tub of granular orchid fertilizer, 'Orchid Focus' style label, small beige/tan pellets visible through a clear window, studio product shot on white background"),
  (9, "granular-fertilizer", 2, "Resealable kraft-paper pouch of granular plant fertilizer, 'Bloom Booster' branding, earthy pellet texture spilling slightly onto the surface, clean white backdrop"),
  (9, "granular-fertilizer", 3, "Cylindrical container of 'NPK 20-20-20' granular fertilizer, bold label with green accents, standing upright, bright even studio lighting"),
  (9, "granular-fertilizer", 4, "Small scoop of granular fertilizer pellets beside a sealed branded bag, 'Root Growth' label, macro close-up texture shot, neutral background"),
  (9, "granular-fertilizer", 5, "Stack of three granular fertilizer tubs in different sizes, 'Green Boost' line, consistent branding, white studio product photography"),
  (10, "liquid-fertilizer", 1, "Amber plastic bottle of liquid orchid fertilizer with a pump/dropper cap, 'Orchid Magic' label, clean white studio background"),
  (10, "liquid-fertilizer", 2, "Clear spray bottle of liquid fertilizer, 'Flower Power' branding, visible pale-yellow liquid inside, bright product photography lighting"),
  (10, "liquid-fertilizer", 3, "Small concentrate bottle of 'Seaweed Extract' liquid fertilizer, dark green bottle with a white label, standing next to its cap, white backdrop"),
  (10, "liquid-fertilizer", 4, "Two liquid fertilizer bottles side by side (different sizes), 'Growth Formula' line, consistent label design, studio lighting on white"),
  (10, "liquid-fertilizer", 5, "Liquid fertilizer bottle being poured into a measuring cap, 'Calcium Plus' label, action-style product shot, clean neutral background"),
  (11, "organic-fertilizer", 1, "Kraft-paper bag of organic plant fertilizer, earthy natural branding, 'Orchid Focus Organic' label, rustic but clean studio backdrop"),
  (11, "organic-fertilizer", 2, "Organic fertilizer pellets/compost in an open burlap sack, natural brown texture, soft natural lighting, neutral background"),
  (11, "organic-fertilizer", 3, "Recycled-paper tub of organic fertilizer, 'Growth Formula' eco-label with green leaf icon, white studio product shot"),
  (11, "organic-fertilizer", 4, "Small organic fertilizer sachet, 'Flower Power' branding on kraft paper, flat lay product photography, clean white background"),
  (11, "organic-fertilizer", 5, "Organic liquid fertilizer bottle in a brown glass bottle, 'Green Boost' natural-style label, warm studio lighting"),
  (12, "pots", 1, "Glossy ceramic orchid pot in white, with drainage holes visible, empty, clean product shot on a white background"),
  (12, "pots", 2, "Woven hanging basket for orchids, natural wood/wicker texture, empty, hanging against a neutral studio backdrop"),
  (12, "pots", 3, "Terracotta orchid pot with slotted sides for root aeration, rustic clay texture, studio lighting, white background"),
  (12, "pots", 4, "Modern black plastic nursery pot with an orchid inside (Phalaenopsis), clean minimalist product shot"),
  (12, "pots", 5, "Set of three ceramic pots in different sizes, stacked or arranged, consistent glaze finish, white studio background"),
  (13, "tools", 1, "Stainless steel pruning shears for orchid care, close-up product shot, sharp clean blades, white background"),
  (13, "tools", 2, "Small plastic fertilizer measuring spoon, branded, flat lay product photography on white"),
  (13, "tools", 3, "Fine-mist spray bottle for misting orchids, clear plastic with a white trigger nozzle, clean studio shot"),
  (13, "tools", 4, "Set of small plant labels/markers (plastic stakes), fanned out, flat lay on a white background"),
  (13, "tools", 5, "Gardening tool set for orchid care - shears, spoon, and mister together, arranged neatly, white studio background"),
  (14, "media", 1, "Bag of sphagnum moss for orchid potting, natural fibrous texture, spilling slightly from an open kraft bag, white background"),
  (14, "media", 2, "Coconut husk chips in a mesh or paper bag, coarse natural brown texture, product shot on white"),
  (14, "media", 3, "Bark chips (orchid bark mix) in a clear plastic bag, chunky brown pieces visible, clean studio lighting"),
  (14, "media", 4, "Pre-mixed 'Orchid Mix' potting medium in a branded bag, blend of bark/moss/perlite visible through a window, white background"),
  (14, "media", 5, "Small pile of growing medium (bark + moss + charcoal mix) poured out next to its packaging, macro texture shot, neutral backdrop"),
]
print(len(PROMPTS), "prompts loaded")

70 prompts loaded


In [4]:
import time

WIDTH, HEIGHT = 1024, 1024
STEPS = 30

for type_num, slug, variant, prompt in PROMPTS:
    fname = f"/content/output/{type_num:02d}-{slug}-{variant}.png"
    if os.path.exists(fname):
        continue
    t0 = time.time()
    image = pipe(
        prompt=prompt + positive_magic,
        negative_prompt=negative_prompt,
        width=WIDTH,
        height=HEIGHT,
        num_inference_steps=STEPS,
        guidance_scale=7.0,
        generator=torch.Generator(device=device).manual_seed(42),
    ).images[0]
    image.save(fname)
    print(f"saved {fname} in {time.time()-t0:.1f}s")

  0%|          | 0/30 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


saved /content/output/01-cattleya-1.png in 33.8s


  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


...

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


saved /content/output/14-media-4.png in 36.4s


  0%|          | 0/30 [00:00<?, ?it/s]

There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


saved /content/output/14-media-5.png in 36.2s


In [5]:
import shutil
from google.colab import files

shutil.make_archive("/content/catalogue_images", "zip", "/content/output")
files.download("/content/catalogue_images.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>